# Silver Layer: Enrich SensorReadings

**Purpose:** Join sendor data with equipment metadata

**Input:**
- `dev.silver.sensor_readings_deduplicated` (sensor readings)
- `dev.bronze.equipment_metadata_raw` (equipment master data)

""Output:"" `dev.silver.sensor_readings_enriched`

**What gets added:**
- equipment_type (CNC_MACHINE, ROBOTIC_ARM, etc.)
- factory_location (Factory_North, Factory_South, etc.)
- production_line (Line_A, Line_B, etc.)
- criticality (HIGH, MEDIUM, LOW)
- install_date (when equipment was installed)

## Configuration

In [0]:
# Configuration
CATALOG = 'dev'

INPUT_SENSOR = f"{CATALOG}.silver.sensor_readings_deduplicated"
INPUT_EQUIPMENT = f"{CATALOG}.bronze.equipment_metadata_raw"
OUTPUT_TABLE = f"{CATALOG}.silver.sensor_readings_enriched"

print(f"Sendor input {INPUT_SENSOR}")
print(f"Equipment input {INPUT_EQUIPMENT}")
print(f"Output: {OUTPUT_TABLE}")

## Load Input Tables

In [0]:
# Read tables
sensor_df = spark.read.table(INPUT_SENSOR)
equipment_df = spark.read.table(INPUT_EQUIPMENT)

print(f"Sensors: {sensor_df.count()} records")
print(f"Equipment: {equipment_df.count()} records")

print(f"\nSensor columns:")
print(sensor_df.columns)

print(f"\nEquipment columns")
print(equipment_df.columns)

## Prepare Dimension Table
Select relevant equipment columns and rename

In [0]:
from pyspark.sql.functions import col

# Select equipment columns to join
equipment_for_join = equipment_df.select(
    col("equipment_id"),
    col("equipment_name"),
    col("equipment_type"),
    col("manufacturer"),
    col("model"),
    col("factory_location"),
    col("production_line"),
    col("status").alias("equipment_status"),
    col("criticality").alias("equipment_criticality"),
    col("install_date")
)

print(f"Equipment dimension prepared: {equipment_for_join.count()} records")
equipment_for_join.show()

## Join Sensor Data with Equipment

In [0]:
# Join on equipment_id
enriched_df = sensor_df.join(
    equipment_for_join,
    on="equipment_id",
    how="left"
)

print(f"After join {enriched_df.count()} records")

# Show sample with enriched data
enriched_df.select(
    "equipment_id",
    "sensor_type",
    "value",
    "equipment_type",
    "factory_location",
    "production_line"
).show(5, truncate=False)

## Check for Umatched Equipment
Verify all sensor readings matched to equipment

In [0]:
from pyspark.sql.functions import col, count, when

# Check for unmatched records (equipment_type is null means no match)
unmatched = enriched_df.filter(col("equipment_type").isNull())
unmatched_count = unmatched.count()

print(f"Unmatched equipment IDs: {unmatched_count}")

if unmatched_count > 0:
    print("\nUnmatched records:")
    unmatched.select("equipment_id", "sensor_type").distinct().show()
    print("\nThese equipment IDs don't exist in equipment_metadata")
else:
    print("All sensor readings matched to equipment!")


## Reorder Columns for Better Orgnization
Put important business columns first

In [0]:
# Select columns in logical order
final_df = enriched_df.select(
    # Key columns
    "equipment_id",
    "equipment_type",
    "factory_location",
    "production_line",

    #Sensor data
    "sensor_type",
    "timestamp",
    "value",
    "unit",

    # Equipment metadata
    "equipment_name",
    "equipment_status",
    "equipment_criticality",
    "manufacturer",
    "model",
    "install_date",

    # Technical metadata
    "sensor_id",
    "data_source",
    "quality_flag",
    "ingestion_timestamp",
    "silver_timestamp",
    "silver_version",
    "dedup_flag"
)

print("Columns reordered for clarity")
print(f"\nFinal schema:")
final_df.printSchema()

## Write to Silver Enriched Table

In [0]:
print(f"Writing {final_df.count()} records to {OUTPUT_TABLE}...")

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Enriched table written!")

## Verify Output Table

In [0]:
# Read back enriched table
enriched_final = spark.read.table(OUTPUT_TABLE)

print(f"Table: {OUTPUT_TABLE}")
print(f"Records: {enriched_final.count()}")

print(f"\nSample enriched data:")
enriched_final.select(
    "equipment_id",
    "equipment_type",
    "sensor_type",
    "value",
    "factory_location",
    "equipment_criticality"
).show(10, truncate=False)

## Enrichment Summary Report

In [0]:
print("=" * 70)
print("ENRICHMENT sUMMARY")
print("=" * 70)
print(f"Input (sensor) : {INPUT_SENSOR} ({sensor_df.count()} records)")
print(f"Input (equipment): {INPUT_EQUIPMENT} ({equipment_df.count()} records)")
print(f"Output: {OUTPUT_TABLE} ({enriched_final.count()} records)")
print(f"")
print(f"ENRICHEMENT RESULTS:")
print(f"Equipment types: {enriched_final.select("equipment_type").distinct().count()}")
print(f"Factories: {enriched_final.select("factory_location").distinct().count()}")
print(f"Production lines: {enriched_final.select("production_line").distinct().count()}")
print(f"Sensor types: {enriched_final.select("sensor_type").distinct().count()}")
print(f"")
print(f"QUALITY CHECK:")
print(f"Unmatched: {unmatched_count}")
print(f"Status {'PASS' if unmatched_count == 0 else 'WARNING'}")
print(f"=" * 70)